In [1]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.stats import binom, beta, poisson
from scipy.special import logsumexp, gamma
from scipy.optimize import minimize
from scipy.integrate import quad
# Can get this plotting library from: https://github.com/aabiddanda/arjun_plot
from arjun_plot.utils import *
from tqdm import tqdm
import polars as pl
import sys
sys.path.append('../code/')
from macgregor import * 

plt.rc('font', family='Arial')

%matplotlib inline

In [2]:
# Some small verification steps to check the density of VAF integrates to 1
for age in np.arange(20,100, 5):
    integrand = quad(lambda x: freq_age_density(s=0.00)(x, age), 0, 0.5)[0]
    assert np.isclose(integrand, 1.0)

/var/folders/l9/4c849x1156qgtv52spq5t1vr0000gn/T/ipykernel_38562/4183896246.py:3: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  integrand = quad(lambda x: freq_age_density(s=0.00)(x, age), 0, 0.5)[0]


In [3]:
tertp_df = pl.read_excel('../data/Table TERTp Age-VAF for Arjun_update 02-06-26 deidentified.xlsx', engine="calamine", read_options={"header_row": 2})
tertp_df.head()

Unique Pt ID,Age at sequencing,TERTp Mutation?,TERTp VAF,Ref_Forward,Ref_Reverse,Alt_F,Alt_R,Total Read Count,De-duplicated total read depth,Ref_F,Ref_R,Alt_F_1,Alt_R_1,Total,Pipeline
str,i64,str,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str
"""523""",30,"""No""",0.0,null,null,null,null,null,null,null,null,null,null,null,null
"""531""",30,"""No""",0.0,null,null,null,null,null,null,null,null,null,null,null,null
"""392""",34,"""No""",0.0,null,null,null,null,null,null,null,null,null,null,null,null
"""526""",34,"""No""",0.0,null,null,null,null,null,null,null,null,null,null,null,null
"""539""",36,"""Yes""",30.3,515,637,233,267,1652,null,null,null,null,null,null,"""Clinical NGS on Research basis…"


In [4]:
# Process data from sheet into numpy arrays for likelihood inference
carrier_df = tertp_df.filter((pl.col('TERTp VAF') > 0) & (pl.col('TERTp VAF') < 50))
noncarrier_df = tertp_df.filter((pl.col('TERTp VAF') == 0))

# Get the carrier information
rs_carriers = np.nan_to_num(carrier_df['Alt_F'].to_numpy()) + np.nan_to_num(carrier_df['Alt_F_1'].to_numpy())
rs_carriers += np.nan_to_num(carrier_df['Alt_R'].to_numpy()) + np.nan_to_num(carrier_df['Alt_R_1'].to_numpy())
ds_carriers = np.nan_to_num(carrier_df['Total Read Count'].to_numpy()) + np.nan_to_num(carrier_df['Total'].to_numpy())
ts_carriers = carrier_df['Age at sequencing'].to_numpy()
assert rs_carriers.size == ts_carriers.size

# Get the non-carrier information
rs_noncarrier = np.zeros(noncarrier_df.shape[0])
ds_noncarrier = np.zeros(noncarrier_df.shape[0])
ts_noncarrier = noncarrier_df['Age at sequencing'].to_numpy()

# concatentate the carrier status
rs_total = np.hstack([rs_carriers, rs_noncarrier])
ds_total = np.hstack([ds_carriers, ds_noncarrier])
ts_total = np.hstack([ts_carriers, ts_noncarrier])

carrier_df.head()

Unique Pt ID,Age at sequencing,TERTp Mutation?,TERTp VAF,Ref_Forward,Ref_Reverse,Alt_F,Alt_R,Total Read Count,De-duplicated total read depth,Ref_F,Ref_R,Alt_F_1,Alt_R_1,Total,Pipeline
str,i64,str,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str
"""539""",36,"""Yes""",30.3,515,637,233,267,1652,null,null,null,null,null,null,"""Clinical NGS on Research basis…"
"""539""",36,"""Yes""",6.6,672,499,56,27,1254,null,null,null,null,null,null,"""Clinical NGS on Research basis…"
"""522""",40,"""Yes""",9.0,1007,1206,95,124,2432,null,null,null,null,null,null,"""Clinical NGS"""
"""391""",45,"""Yes""",6.32,360,277,34,9,680,null,null,null,null,null,null,"""Clinical NGS"""
"""529""",47,"""Yes""",2.36,506,322,15,5,848,null,null,null,null,null,null,"""Clinical NGS"""


In [5]:
# NOTE: will need to downsample some of the higher coverage individuals ... 
macgregor_total_loglik(rs_total,ds_total,ts_total, s=0.01, mu=1e-7)

np.float64(-1244.2287566208036)

In [6]:
# NOTE: this implements an MLE solution using a 
mle_inference_sonly(rs_total,ds_total,ts_total, mu=1e-9)

/Users/aabiddanda/Workdir/shortTL_vaf_modeling/notebooks/../code/macgregor.py:35: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  integrand = quad(lambda f: binom.pmf(r2, n=d2, p=f) * x(f,t), 0, 0.5)
/Users/aabiddanda/Workdir/shortTL_vaf_modeling/notebooks/../code/macgregor.py:43: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  integrand = quad(lambda f: freq_age_density(s=s, mu=mu, **kwargs)(f,t), 0.0, f0)
/Users/aabiddanda/Workdir/shortTL_vaf_modeling/notebooks/../code/macgregor.py:55: RuntimeWarning: invalid value encountered in log
  loglik += np.log(macgregor_single_likelihood(r,d, t=t, s=s, mu=mu)[0])



Optimization terminated successfully;
The returned value satisfies the termination criteria
(using xtol =  1e-05 )


 message: Solution found.
 success: True
  status: 0
     fun: 268.9489564871042
       x: 0.4273995042173249
     nit: 23
    nfev: 23

In [13]:
f = lambda x: -macgregor_total_loglik(rs_total, ds_total, ts_total, mu=x[0], s=x[1])
opt_res = minimize(f, x0=[1e-9, 1e-2], method='Powell', bounds=[(1e-20, 1e-6), (1e-10, 0.5)], options={'disp': True})

Optimization terminated successfully.
         Current function value: 163.907403
         Iterations: 2
         Function evaluations: 38


In [14]:
opt_res

 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 163.9074031960921
       x: [ 3.820e-07  2.392e-01]
     nit: 2
   direc: [[ 1.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00]]
    nfev: 38